In [1]:
# Homework 3, Question 2C: Numerical gradient verification
# Dependencies: numpy, autograd (pip install numpy autograd)
import numpy as np
import autograd.numpy as anp
from autograd import grad
from importlib.metadata import version

x = np.float64(0.7)
y = np.float64(1.0)
weights = np.array([0.2, -0.3, 0.4, -0.5], dtype=np.float64)
k = np.float64(1.0)
epsilon = np.float64(1e-5)

print("NumPy:", version("numpy"))
print("Autograd:", version("autograd"))
print("Arithmetic:", weights.dtype)

import os
import platform
import time

cpu = platform.processor()
if os.path.exists("/proc/cpuinfo"):
    with open("/proc/cpuinfo") as f:
        for line in f:
            if line.startswith("model name"):
                cpu = line.split(":", 1)[1].strip()
                break

print("Operating system:", platform.platform())
print("Python:", platform.python_version())
print("CPU:", cpu or "Unknown")
print("Logical CPUs:", os.cpu_count())
if hasattr(os, "sched_getaffinity"):
    print("Logical CPUs available to this process:", len(os.sched_getaffinity(0)))
print("Device: CPU")

NumPy: 1.26.4
Autograd: 1.9.1
Arithmetic: float64
Operating system: Linux-7.0.0-31-generic-x86_64-with-glibc2.39
Python: 3.12.3
CPU: Intel(R) Core(TM) i7-8650U CPU @ 1.90GHz
Logical CPUs: 8
Logical CPUs available to this process: 8
Device: CPU


In [2]:
cell_start = time.perf_counter()

def sigmoid(u):
    return 1.0 / (1.0 + anp.exp(-u))


def forward(w, activation):
    w1, w2, w3, w4 = w
    if activation == "sigmoid":
        h1 = sigmoid(w1 * x)
        h2 = sigmoid(w2 * x)
    else:
        h1 = anp.logaddexp(0.0, 2.0 * k * w1 * x) / (2.0 * k)
        h2 = anp.logaddexp(0.0, 2.0 * k * w2 * x) / (2.0 * k)
    z = w3 * h1 + w4 * h2
    return h1, h2, z


def loss(w, activation):
    _, _, z = forward(w, activation)
    # Stable binary cross-entropy from logits.
    return anp.logaddexp(0.0, z) - y * z

print(f"Cell runtime: {time.perf_counter() - cell_start:.6f} seconds")

Cell runtime: 0.000003 seconds


In [3]:
cell_start = time.perf_counter()

def analytic_gradients(w, activation):
    w1, w2, w3, w4 = w
    h1, h2, z = forward(w, activation)
    delta = sigmoid(z) - y
    if activation == "sigmoid":
        d1 = h1 * (1.0 - h1)
        d2 = h2 * (1.0 - h2)
    else:
        d1 = sigmoid(2.0 * k * w1 * x)
        d2 = sigmoid(2.0 * k * w2 * x)
    return np.array([
        delta * w3 * d1 * x,
        delta * w4 * d2 * x,
        delta * h1,
        delta * h2,
    ], dtype=np.float64)


def finite_differences(w, activation):
    gradients = np.zeros_like(w)
    for i in range(len(w)):
        plus = w.copy()
        minus = w.copy()
        plus[i] += epsilon
        minus[i] -= epsilon
        gradients[i] = (loss(plus, activation) - loss(minus, activation)) / (2.0 * epsilon)
    return gradients

print(f"Cell runtime: {time.perf_counter() - cell_start:.6f} seconds")

Cell runtime: 0.000003 seconds


In [4]:
cell_start = time.perf_counter()

results = {}
for activation in ["sigmoid", "softplus"]:
    analytic = analytic_gradients(weights, activation)
    automatic = grad(loss)(weights, activation)
    numerical = finite_differences(weights, activation)
    ad_error = np.max(np.abs(analytic - automatic))
    fd_error = np.max(np.abs(analytic - numerical))
    results[activation] = {
        "analytic": analytic,
        "automatic": automatic,
        "finite_difference": numerical,
        "max_ad_discrepancy": ad_error,
        "max_fd_discrepancy": fd_error,
    }

    print("\n" + ("Sigmoid" if activation == "sigmoid" else "Scaled Softplus (k = 1)"))
    print(f"{'Weight':<8} {'Analytic':>18} {'Autodiff':>18} {'Finite difference':>18}")
    for i in range(4):
        print(f"w{i + 1:<7} {analytic[i]:18.12f} {automatic[i]:18.12f} {numerical[i]:18.12f}")
    print(f"Maximum |analytic - autodiff|: {ad_error:.6e}")
    print(f"Maximum |analytic - finite difference|: {fd_error:.6e}")

    np.testing.assert_allclose(analytic, automatic, rtol=0, atol=1e-12)
    np.testing.assert_allclose(analytic, numerical, rtol=0, atol=1e-8)

print(f"Cell runtime: {time.perf_counter() - cell_start:.6f} seconds")


Sigmoid
Weight             Analytic           Autodiff  Finite difference
w1          -0.035000918963    -0.035000918963    -0.035000918963
w2           0.043484696482     0.043484696482     0.043484696477
w3          -0.268791282082    -0.268791282082    -0.268791282082
w4          -0.224950589689    -0.224950589689    -0.224950589683
Maximum |analytic - autodiff|: 5.551115e-17
Maximum |analytic - finite difference|: 5.799611e-12

Scaled Softplus (k = 1)
Weight             Analytic           Autodiff  Finite difference
w1          -0.078049347229    -0.078049347229    -0.078049347224
w2           0.067922216435     0.067922216435     0.067922216440
w3          -0.206270074531    -0.206270074531    -0.206270074532
w4          -0.123587760488    -0.123587760488    -0.123587760492
Maximum |analytic - autodiff|: 0.000000e+00
Maximum |analytic - finite difference|: 5.580245e-12
Cell runtime: 0.028791 seconds
